# mixed ships near the Seattle ferry terminal and stadiums

ships of three sizes, with building contact, grounding, and ship contact

run `seattle_shoreline.ipynb` first; this reuses its 24 m flow grid

In [ ]:
from pathlib import Path
import hashlib
import json
import sys

import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as PolygonPatch
from netCDF4 import Dataset
import numpy as np
import pandas as pd
from shapely.geometry import box, shape
from shapely.ops import transform, unary_union
from shapely.prepared import prep
from scipy.ndimage import distance_transform_edt
from IPython.display import Video, display

HERE = Path.cwd().resolve()
if not (HERE / 'setrun.py').exists(): HERE = HERE / 'examples' / 'seattle_debris'
if not (HERE / 'setrun.py').exists(): raise FileNotFoundError('Run from the Seattle example or repository root')
sys.path.insert(0, str(HERE.parents[1] / 'src'))
from geoclaw_debris.ship_tracking import Flow, Ship, check_pose, hull_polygon, simulate_fleet

FLOW_FILE = HERE / '_output' / 'seattle_shoreline' / 'seattle_fault_fgout.nc'
OUT = HERE / '_output' / 'seattle_harbor'
OUT.mkdir(parents=True, exist_ok=True)
if not FLOW_FILE.exists(): raise FileNotFoundError('Run seattle_shoreline.ipynb first to create ' + str(FLOW_FILE))

BOUNDS = [-122.365, -122.330, 47.583, 47.606]
ships = [Ship(length_m=30., beam_m=8., draft_m=1.5),
         Ship(length_m=60., beam_m=14., draft_m=3.),
         Ship(length_m=120., beam_m=24., draft_m=5.)]
names = ['small boat', 'medium ship', 'large ship']
colors = ['#e6a22a', '#d84b45', '#7145ac']
HEADING = 90.
DT = 1.


In [ ]:
# starting positions
inventory_file = HERE.parents[1] / 'SeattleBuildings' / 'Seattle_BuildingInventory.geojson'
features = json.loads(inventory_file.read_text())['features']
with Dataset(FLOW_FILE) as ds:
    assert ds.inventory_sha256 == hashlib.sha256(inventory_file.read_bytes()).hexdigest(), 'Building inventory changed; rerun seattle_shoreline.ipynb'
    flow = Flow(ds)
    domain = box(flow.lon.min(), flow.lat.min(), flow.lon.max(), flow.lat.max())
    footprints = [shape(f['geometry']) for f in features]
    footprints = [p for p in footprints if p.intersects(domain)]
    buildings = unary_union(footprints)
    obstacles = prep(transform(flow.to_xy, buildings))
    metric_footprints = [transform(flow.to_xy, p) for p in footprints]
    gx, gy = flow.to_xy(flow.lon, flow.lat)
    initial_depth = np.asarray(ds['h'][0])
    water_times = np.asarray(ds['time'][:])
    distance = distance_transform_edt(initial_depth > .1, sampling=(gx[1]-gx[0], gy[1]-gy[0]))
    view = box(BOUNDS[0], BOUNDS[2], BOUNDS[1], BOUNDS[3])
    metric_view = transform(flow.to_xy, view)
    candidates = np.argwhere((distance >= 40.) & (distance <= 250.))
    rng = np.random.default_rng(23)
    fleet, starts, starting_hulls = [], [], []
    for kind in [2, 1, 0]:
        count = 0
        for i, j in rng.permutation(candidates):
            center = (gx[i], gy[j])
            hull = hull_polygon(center, np.deg2rad(HEADING), ships[kind])
            if not metric_view.contains(hull): continue
            clear, _, _ = check_pose(flow, obstacles, ships[kind], center, np.deg2rad(HEADING), water_times[0])
            if clear and all(not hull.buffer(15.).intersects(other) for other in starting_hulls):
                fleet.append(kind)
                starts.append(flow.to_lonlat(*center))
                starting_hulls.append(hull)
                count += 1
                if count == 4: break
        if count < 4: raise RuntimeError(f'Only placed {count} {names[kind]} vessels; widen the view or shore-distance range')

    # put one small boat near a large ship
    boat = fleet.index(0)
    large = fleet.index(2)
    x, y = flow.to_xy(*starts[large])
    center = (x-40., y)
    hull = hull_polygon(center, np.deg2rad(HEADING), ships[0])
    clear, _, _ = check_pose(flow, obstacles, ships[0], center, np.deg2rad(HEADING), water_times[0])
    assert clear and metric_view.contains(hull)
    assert all(not hull.buffer(15.).intersects(other) for k, other in enumerate(starting_hulls) if k != boat)
    starts[boat] = flow.to_lonlat(*center)
    starting_hulls[boat] = hull

pd.DataFrame([dict(ship=k, kind=names[kind], length_m=ships[kind].length_m,
                   beam_m=ships[kind].beam_m, draft_m=ships[kind].draft_m,
                   lon=starts[k][0], lat=starts[k][1]) for k, kind in enumerate(fleet)])


In [ ]:
def harbor_map():
    fig, ax = plt.subplots(figsize=(10, 9), constrained_layout=True)
    ax.contourf(gx, gy, initial_depth.T, levels=[-.01, .1, float(initial_depth.max())+1], colors=['#e5e0d6', '#dceaf0'])
    for footprint in metric_footprints:
        for piece in getattr(footprint, 'geoms', [footprint]):
            if piece.geom_type == 'Polygon':
                ax.add_patch(PolygonPatch(np.asarray(piece.exterior.coords), facecolor='#827e76', edgecolor='none', alpha=.7))
    x, y = flow.to_xy(np.array(BOUNDS[:2]), np.array(BOUNDS[2:]))
    ax.set(xlim=x, ylim=y, xlabel='east (m)', ylabel='north (m)', aspect='equal')
    for lon, lat, label in [(-122.337, 47.602, 'ferry terminal'), (-122.334, 47.591, 'stadiums')]:
        ax.text(*flow.to_xy(lon, lat), label, fontsize=9)
    for kind in range(3): ax.plot([], [], color=colors[kind], lw=5, label=f'{names[kind]} ({ships[kind].length_m:.0f} m)')
    ax.legend(loc='lower left')
    return fig, ax


## trajectories

In [ ]:
with Dataset(FLOW_FILE) as ds:
    flow = Flow(ds)
    paths = simulate_fleet(flow, buildings, [ships[kind] for kind in fleet], starts,
                           heading_deg=HEADING, start_time=water_times[0],
                           end_time=water_times[-1], dt=DT)

ship_times = np.array([r['time_s'] for r in paths[0]])
assert np.isin(water_times, ship_times).all(), 'DT must align with the saved flow times'
for rows in zip(*paths):
    outlines = [hull_polygon((r['x_m'], r['y_m']), np.deg2rad(r['heading_deg']), ships[kind])
                for r, kind in zip(rows, fleet)]
    assert all(not obstacles.intersects(hull) for hull in outlines)
    assert all(not outlines[i].intersects(outlines[j]) for i in range(len(outlines)) for j in range(i))

tracks = pd.concat([pd.DataFrame(path).assign(ship=k, kind=names[fleet[k]]) for k, path in enumerate(paths)], ignore_index=True)
tracks.to_csv(OUT / 'harbor_paths.csv', index=False)
assert np.isfinite(tracks[['x_m', 'y_m', 'heading_deg', 'u_mps', 'v_mps']]).all().all()
summary = []
for k, path in enumerate(paths):
    p = pd.DataFrame(path)
    contact = p[p.status != 'afloat']
    summary.append(dict(ship=k, kind=names[fleet[k]], final_status=p.status.iloc[-1],
                        first_contact_s=contact.time_s.iloc[0] if len(contact) else np.nan,
                        travel_m=np.hypot(np.diff(p.x_m), np.diff(p.y_m)).sum(),
                        minimum_depth_m=p.min_depth_m.min(),
                        sliding_steps=int((p.status == 'sliding').sum()),
                        blocked_steps=int((p.status == 'blocked').sum()),
                        ship_contact_steps=int((p.status == 'ship contact').sum()),
                        first_ship_contact_s=next((r['time_s'] for r in path if r['status'] == 'ship contact'), np.nan)))
summary = pd.DataFrame(summary)
summary.to_csv(OUT / 'harbor_summary.csv', index=False)
display(summary.round(2))


In [ ]:
fig, ax = harbor_map()
for k, path in enumerate(paths):
    p = pd.DataFrame(path)
    color = colors[fleet[k]]
    ax.plot(p.x_m, p.y_m, color=color, lw=1)
    ax.plot(p.x_m.iloc[0], p.y_m.iloc[0], 'o', color=color, ms=3)
    hull = hull_polygon((p.x_m.iloc[-1], p.y_m.iloc[-1]), np.deg2rad(p.heading_deg.iloc[-1]), ships[fleet[k]])
    ax.add_patch(PolygonPatch(np.asarray(hull.exterior.coords), facecolor=color, edgecolor='#222'))
    ax.text(p.x_m.iloc[-1], p.y_m.iloc[-1], str(k), fontsize=8)
ax.set_title('full trajectories: dots are starts, hulls are final positions')
fig.savefig(OUT / 'trajectories.png', dpi=150)
plt.show()


## animation

arrows show water velocity; shading shows water depth; orange outlines mark ship contact.

In [ ]:
from matplotlib.animation import FuncAnimation, FFMpegWriter
import matplotlib as mpl
import shutil

with Dataset(FLOW_FILE) as ds:
    depth = np.asarray(ds['h'][:])
    u, v = np.asarray(ds['u'][:]), np.asarray(ds['v'][:])

fig, ax = harbor_map()
water = ax.pcolormesh(gx, gy, np.ma.masked_less(depth[0].T, .1), cmap='Blues', vmin=0, vmax=12, shading='auto', alpha=.6)
fig.colorbar(water, ax=ax, label='water depth (m)', shrink=.65)
arrows = ax.quiver(gx[::4], gy[::4], u[0, ::4, ::4].T, v[0, ::4, ::4].T, angles='xy', scale_units='xy', scale=.06, color='#36576b')
ax.quiverkey(arrows, .08, .96, 2., '2 m/s', labelpos='E')
hulls, trails = [], []
for kind in fleet:
    hull = PolygonPatch(np.zeros((4, 2)), facecolor=colors[kind], edgecolor='#222', zorder=5)
    ax.add_patch(hull)
    hulls.append(hull)
    trails.append(ax.plot([], [], color=colors[kind], lw=1, zorder=4)[0])
xy = [np.array([[r['x_m'], r['y_m']] for r in path]) for path in paths]

def update(frame):
    time = water_times[frame]
    step = np.searchsorted(ship_times, time)
    water.set_array(np.ma.masked_less(depth[frame].T, .1).ravel())
    arrows.set_UVC(np.ma.masked_where(depth[frame, ::4, ::4].T < .1, u[frame, ::4, ::4].T),
                   np.ma.masked_where(depth[frame, ::4, ::4].T < .1, v[frame, ::4, ::4].T))
    for k, path in enumerate(paths):
        hull = hull_polygon(xy[k][step], np.deg2rad(path[step]['heading_deg']), ships[fleet[k]])
        hulls[k].set_xy(np.asarray(hull.exterior.coords))
        contact = path[step]['status'] == 'ship contact'
        hulls[k].set_edgecolor('#ff8c00' if contact else '#222')
        hulls[k].set_linewidth(2. if contact else 1.)
        trails[k].set_data(xy[k][:step+1, 0], xy[k][:step+1, 1])
    contacts = sum(path[step]['status'] == 'ship contact' for path in paths)
    ax.set_title(f'mixed harbor fleet: {time:.0f} s after earthquake; {contacts} ships in contact')

frames = np.unique(np.r_[np.arange(0, len(water_times), 4), len(water_times)-1])
animation = FuncAnimation(fig, update, frames=frames, interval=125)
plt.close(fig)
ffmpeg = shutil.which('ffmpeg')
if ffmpeg is None:
    local_ffmpeg = HERE.parents[1] / '.venv' / 'bin' / 'ffmpeg'
    if local_ffmpeg.is_file(): ffmpeg = str(local_ffmpeg)
if ffmpeg is None: raise RuntimeError('Install ffmpeg to export the animation')
movie = OUT / 'harbor_fleet.mp4'
with mpl.rc_context({'animation.ffmpeg_path': ffmpeg}):
    animation.save(str(movie), writer=FFMpegWriter(fps=8), dpi=100)
Video(str(movie), embed=True)
